In [ ]:
# ============================================================
# 1. Install
# ============================================================

!pip install -q -U ultralytics

import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ============================================================
# 2. Path setting
# ============================================================

DRIVE_BASE_DIR = Path(
    "/content/drive/MyDrive/SAR_AI_Ship_Detection/LS-SSDD"
)

LOCAL_BASE_DIR = Path("/content/LS-SSDD")

if not LOCAL_BASE_DIR.exists():
    shutil.copytree(DRIVE_BASE_DIR, LOCAL_BASE_DIR)
    print("Copied LS-SSDD to /content")
else:
    print("LS-SSDD already exists in /content")

BASE_DIR = LOCAL_BASE_DIR

ANN_DIR = BASE_DIR / "Annotations_sub"

In [ ]:
# ============================================================
# Convert LS-SSDD → YOLO
# ============================================================

train_ids = read_ids(TRAIN_TXT)
val_ids   = read_ids(VAL_TXT)
test_ids  = read_ids(TEST_TXT)

prepare_split(train_ids, "train")
prepare_split(val_ids, "val")
prepare_split(test_ids, "test")

In [ ]:
# ============================================================
# Make data.yaml
# ============================================================

data_yaml = OUT_DIR / "data.yaml"

with open(data_yaml, "w") as f:
    f.write(f"""
path: {OUT_DIR}

train: images/train
val: images/val
test: images/test

names:
  0: ship
""")

print(data_yaml.read_text())

In [ ]:
# ============================================================
# 6. Fine-tune HRSID best model on LS-SSDD
# ============================================================

HRSID_BEST_MODEL = Path(
    "/content/drive/MyDrive/"
    "SAR_AI_Ship_Detection/"
    "trained_models/"
    "hrsid_yolo26s_20epoch_best.pt"
)

assert HRSID_BEST_MODEL.exists(), \
    f"Model not found: {HRSID_BEST_MODEL}"

print("HRSID pretrained model:")
print(HRSID_BEST_MODEL)

# HRSID에서 학습된 YOLO26s weight 불러오기
model = YOLO(str(HRSID_BEST_MODEL))

# LS-SSDD로 fine-tuning
model.train(
    data=str(data_yaml),

    epochs=20,

    imgsz=800,
    batch=16,

    device=0,
    workers=2,

    project="/content/runs",
    name="ls_ssdd_yolo26s_hrsid_finetune",

    pretrained=True
)

In [ ]:
# ============================================================
# 7. Load fine-tuned BEST model
# ============================================================

FINETUNED_BEST = Path(
    "/content/runs/"
    "ls_ssdd_yolo26s_hrsid_finetune/"
    "weights/best.pt"
)

assert FINETUNED_BEST.exists(), \
    f"Model not found: {FINETUNED_BEST}"

best_model = YOLO(str(FINETUNED_BEST))

print("Fine-tuned best model:")
print(FINETUNED_BEST)

In [ ]:
# ============================================================
# 8. Validation
# ============================================================

val_metrics = best_model.val(
    data=str(data_yaml),
    split="val",
    imgsz=800,
    device=0
)

print("=" * 50)
print("Validation Results")
print("=" * 50)

print(f"Precision      : {val_metrics.box.mp:.4f}")
print(f"Recall         : {val_metrics.box.mr:.4f}")
print(f"mAP@0.5        : {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {val_metrics.box.map:.4f}")

In [ ]:
# ============================================================
# 9. Test
# ============================================================

test_metrics = best_model.val(
    data=str(data_yaml),
    split="test",
    imgsz=800,
    device=0
)

print("=" * 50)
print("Test Results")
print("=" * 50)

print(f"Precision      : {test_metrics.box.mp:.4f}")
print(f"Recall         : {test_metrics.box.mr:.4f}")
print(f"mAP@0.5        : {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {test_metrics.box.map:.4f}")

In [ ]:
# ============================================================
# 10. Save fine-tuned model to Google Drive
# ============================================================

RUN_DIR = Path(
    "/content/runs/"
    "ls_ssdd_yolo26s_hrsid_finetune/"
    "weights"
)

SAVE_DIR = Path(
    "/content/drive/MyDrive/"
    "SAR_AI_Ship_Detection/"
    "trained_models"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_SAVE_PATH = (
    SAVE_DIR /
    "ls_ssdd_yolo26s_hrsid_finetune_20epoch_best.pt"
)

LAST_SAVE_PATH = (
    SAVE_DIR /
    "ls_ssdd_yolo26s_hrsid_finetune_20epoch_last.pt"
)

shutil.copy(
    RUN_DIR / "best.pt",
    BEST_SAVE_PATH
)

shutil.copy(
    RUN_DIR / "last.pt",
    LAST_SAVE_PATH
)

print("Saved:")
print(BEST_SAVE_PATH)
print(LAST_SAVE_PATH)

print()
print("Best size:",
      BEST_SAVE_PATH.stat().st_size / 1024 / 1024,
      "MB")